# Benchmark - inference notebook
> Build dataset and use it to compute structured data extraction from screenshots of the scalable broker and generate performance indicators.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import time
from enum import StrEnum, auto
from pathlib import Path
from typing import TypedDict, cast

import httpx
import instructor
import ipywidgets as widgets
import polars as pl
import tqdm
from instructor.processing.multimodal import Image as InstructorImage
from IPython.display import clear_output
from openai import OpenAI
from openai.types.chat.chat_completion import ChatCompletion
from PIL import Image, ImageFile
from plotnine import aes, geom_histogram, geom_point, ggplot
from pydantic import BaseModel

from fintl.common import Config
from fintl.common.extraction.context import (
    _SYSTEM_PROMPT,
    _BalanceInfoExtract,
)
from fintl.common.extraction.errors import OllamaInferenceError
from fintl.common.extraction.ollama import _get_client


class ModelProvider(StrEnum):
    """Model provider enum eheh."""

    ollama = auto()
    llama_swap = auto()

In [ ]:
N_RUNS = 5
PERFORM_EXTRACTIONS = True
RESULTS_DIR = Path("./benchmark-results")

TIMEOUT = 2 * 60
LLAMA_SWAP_URL = "http://0.0.0.0:8080"
OLLAMA_URL = "http://localhost:11434"

# PROVIDER = ModelProvider.ollama
# MODEL = "gemma4:latest"

PROVIDER = ModelProvider.llama_swap
# MODEL = "qwen-3.6-27b"
MODEL = "gemma-4-31b"

PROVIDER, MODEL

In [ ]:
RESULTS_DIR.mkdir(exist_ok=True)

## Build dataset

In [ ]:
config = Config()

In [ ]:
config.target_dir

In [ ]:
root_dir = config.target_dir / "Scalable/broker/broker20260309"
assert root_dir.exists()

In [ ]:
images_dir = root_dir / "raw"
assert images_dir.exists()

In [ ]:
balances = pl.read_parquet(root_dir / "balances.parquet")
balances.head(2)

In [ ]:
balances["file"].item(0)

In [ ]:
class Correction(BaseModel):
    """A single index/amount correction to apply to the dataset."""

    idx: int
    amount: float


class Corrections(BaseModel):
    """A collection of index/amount corrections loaded from a JSON file."""

    entries: list[Correction]

    def get_map(self) -> dict[int, float]:
        """Return a mapping from index to corrected amount."""
        return {e.idx: e.amount for e in self.entries}


path_corrections = Path("./corrections.json")
with path_corrections.open() as f:
    display(Corrections.model_validate_json(f.read()))  # ty: ignore[unresolved-reference]

In [ ]:
class ScreenshotDataset:
    """Dataset of broker screenshots paired with ground-truth balance amounts."""

    df: pl.DataFrame
    idx_col: str
    path_col: str
    target_col: str
    path_corrections: Path | None = None
    corrections: Corrections | None = None
    correction_applied: bool = False

    def __init__(
        self,
        df: pl.DataFrame,
        idx_col: str = "__idx",
        path_col: str = "file",
        target_col: str = "amount",
        path_corrections: Path | None = None,
    ):
        """Initialise the dataset, optionally loading and applying target-value corrections."""
        self.df = df.with_row_index(idx_col)
        self.idx_col = idx_col
        self.path_col = path_col
        self.target_col = target_col
        self.path_corrections = path_corrections

        if self.path_corrections and not self.correction_applied:
            self._correct_target_values()

    def __len__(self):
        """Return the number of unique samples in the dataset."""
        return len(self.df)

    def __getitem__(self, idx: int) -> tuple[ImageFile.ImageFile, Path, float]:
        """Return (image, path, target_amount) for the sample at *idx*."""
        idxs = self.df[self.idx_col].unique()
        if not 0 <= idx < len(self):
            msg = f"{idx=} is not a valid value. Select one in [{min(idxs)}, {max(idxs)}]"
            raise IndexError(msg)

        row = self.df.filter(pl.col(self.idx_col) == idx)
        path = Path(row[self.path_col].item())
        img = Image.open(path)
        target = row[self.target_col].item()
        return img, path, target

    def _correct_target_values(self):
        if self.path_corrections is None:
            msg = "Can only assign correct target values if path_corrections was passed."
            raise ValueError(msg)

        print("Applying corrections")
        with self.path_corrections.open() as f:
            self.corrections = Corrections.model_validate_json(f.read())

        target_values = self.df[self.target_col].to_list()
        for idx, correction in self.corrections.get_map().items():
            target_values[idx] = correction

        self.df = self.df.with_columns(**{self.target_col: pl.Series(target_values)})

    def __iter__(self):
        """Iterate over all (image, path, target_amount) samples in order."""
        for idx in range(len(self)):
            yield self[idx]


ds = ScreenshotDataset(balances, path_corrections=path_corrections)

In [ ]:
_img, _path, _amount = ds[17]
_img

In [ ]:
def make_viewer(ds: ScreenshotDataset):
    """Build and display an interactive ipywidgets viewer for the dataset."""
    idx = {"current": 0}

    out = widgets.Output()
    label = widgets.Label(value=f"Sample 0 / {len(ds) - 1}")
    prev_btn = widgets.Button(description="◀ Prev")
    next_btn = widgets.Button(description="Next ▶")

    def show(i: int):
        img, path, target = ds[i]
        label.value = f"Sample {i} / {len(ds) - 1}"
        with out:
            clear_output(wait=True)
            display(img)  # ty: ignore[unresolved-reference]
            print(f"target : {target}")
            print(f"path   : {path}")

    def on_prev(_):
        idx["current"] = max(0, idx["current"] - 1)
        show(idx["current"])

    def on_next(_):
        idx["current"] = min(len(ds) - 1, idx["current"] + 1)
        show(idx["current"])

    prev_btn.on_click(on_prev)
    next_btn.on_click(on_next)

    show(0)
    display(widgets.VBox([label, widgets.HBox([prev_btn, next_btn]), out]))  # ty: ignore[unresolved-reference]


make_viewer(ds)

## Perform inference / extraction

In [ ]:
def check_model_availability(
    model_name: str, *, base_url: str = "http://localhost:11434", timeout: int = 2 * 60
):
    """Verify that *model_name* is installed and responsive in the local ollama instance."""
    with httpx.Client(base_url=base_url, timeout=timeout) as client:
        # 1. Sanity check: is the Ollama server up?
        try:
            resp = client.get("/")
            resp.raise_for_status()
        except httpx.ConnectError:
            print("ERROR: Could not connect to Ollama at", base_url)
            sys.exit(1)

        # 2. Verify the model is available locally
        models_resp = client.get("/api/tags")
        models_resp.raise_for_status()
        installed = [m["name"] for m in models_resp.json().get("models", [])]

        if model_name not in installed:
            print(f"ERROR: Model '{model_name}' not found.")
            print("Available models:", installed if installed else "(none)")
            sys.exit(1)

        print(f"OK: '{model_name}' is available.")

        # 3. Quick inference sanity check
        gen_resp = client.post(
            "/api/generate",
            json={
                "model": model_name,
                "prompt": "Say hello in one short sentence.",
                "stream": False,  # get the full response at once
                "options": {"temperature": 0.0},
            },
        )
        gen_resp.raise_for_status()
        answer = gen_resp.json()["response"].strip()
        print(f"Model replied: {answer}")


def check_health(client: httpx.Client) -> bool:
    """GET /health — llama-swap-native, returns 'OK' on 200."""
    r = client.get("/health", timeout=5.0)
    r.raise_for_status()
    body = r.text.strip()
    print(f"  /health -> {r.status_code} '{body}'")
    return r.status_code == 200 and body == "OK"


def check_model_available(client: httpx.Client, model: str) -> bool:
    """GET /v1/models — OpenAI-compatible list; look for our model id in data[].id."""
    r = client.get("/v1/models")
    r.raise_for_status()
    payload = r.json()
    ids = [m["id"] for m in payload.get("data", [])]
    print(f"  /v1/models -> {len(ids)} model(s): {ids}")
    return model in ids


def check_inference(client: httpx.Client, model: str) -> bool:
    """POST /v1/chat/completions with a tiny hello-world prompt.

    Using chat/completions (not /v1/completions) because:
      - it's the endpoint instructor + most clients use,
      - the 'model' field is what triggers llama-swap's hot-swap,
      - omitting 'model' makes the proxy refuse to forward.
    """
    r = client.post(
        "/v1/chat/completions",
        json={
            "model": model,  # required — drives the swap
            "messages": [{"role": "user", "content": "Say hello in one word."}],
            "max_tokens": 2000,
            "temperature": 0,
        },
    )
    r.raise_for_status()
    content = r.json()["choices"][0]["message"]["content"]
    print(f"  /v1/chat/completions -> '{content}'")
    return bool(content)


def sanity_check(model: str, *, timeout: int = TIMEOUT, base_url: str = LLAMA_SWAP_URL) -> bool:
    """Run health, model-availability, and inference checks against the llama-swap server."""
    print(f"llama-swap sanity check @ {base_url} (model='{model}')")
    with httpx.Client(base_url=base_url, timeout=timeout) as client:
        try:
            assert check_health(client), "health check failed"
            assert check_model_available(client, model), f"model '{model}' not in /v1/models"
            assert check_inference(client, model), "inference produced no content"
        except (httpx.HTTPError, AssertionError, KeyError) as e:
            print(f"  ✗ FAILED: {e}")
            return False
    print("  ✓ All checks passed")
    return True


match PROVIDER:
    case ModelProvider.ollama:
        check_model_availability(MODEL, timeout=TIMEOUT, base_url=OLLAMA_URL)
    case ModelProvider.llama_swap:
        sanity_check(MODEL, timeout=TIMEOUT, base_url=LLAMA_SWAP_URL)

In [ ]:
ExtractionResponse = tuple[_BalanceInfoExtract, ChatCompletion]


def _get_ollama_extraction(
    file_path: Path, extraction_client: instructor.Instructor, timeout: int
) -> ExtractionResponse:
    """Run LM inference using ollama to extract balance information from an image file."""
    from instructor.core.exceptions import InstructorRetryException

    try:
        res = extraction_client.create_with_completion(  # type: ignore
            response_model=_BalanceInfoExtract,
            messages=[
                {"role": "system", "content": _SYSTEM_PROMPT},
                {
                    "role": "user",
                    "content": [
                        "Please extract data from the following image",
                        InstructorImage.from_path(file_path),
                    ],
                },  # type: ignore[arg-type]
            ],
            timeout=timeout,
        )

        return cast(ExtractionResponse, res)
    except InstructorRetryException as exc:
        last = exc.failed_attempts[-1].exception if exc.failed_attempts else exc
        # explicitly cutting of the traceback here for readability.
        # remove `from None` if you need to debug.
        raise OllamaInferenceError(
            f"Ollama inference failed for {file_path.name}: {last}"
        ) from None

    else:
        msg = "No idea how we got here, but the _get_lm_extraction failed."
        raise RuntimeError(msg)


class InferenceError(Exception):
    """Raised when the inference fails."""


def _get_llama_swap_extraction(
    file_path: Path, extraction_client: instructor.Instructor, model: str, timeout: int
) -> ExtractionResponse:
    """Run LM inference to extract balance information from an image file."""
    from instructor.core.exceptions import InstructorRetryException

    try:
        res = extraction_client.create_with_completion(  # type: ignore
            model=model,
            response_model=_BalanceInfoExtract,
            messages=[
                {"role": "system", "content": _SYSTEM_PROMPT},
                {
                    "role": "user",
                    "content": [
                        "Please extract data from the following image",
                        InstructorImage.from_path(file_path),
                    ],
                },  # type: ignore[arg-type]
            ],
            timeout=timeout,
        )

        return cast(ExtractionResponse, res)
    except InstructorRetryException as exc:
        last = exc.failed_attempts[-1].exception if exc.failed_attempts else exc
        raise InferenceError(f"llama-swap inference failed for {file_path.name}: {last}") from None

    else:
        msg = "No idea how we got here, but the _get_lm_extraction failed."
        raise RuntimeError(msg)


class ExtractionOutput(BaseModel):
    """Container for the result of a single extraction attempt."""

    extraction: _BalanceInfoExtract | None
    completion: ChatCompletion | None
    elapsed: float
    ok: bool
    error_message: str


class OllamaExtractionModel:
    """Extraction model that delegates inference to a local ollama instance."""

    model: str
    base_url: str
    client: instructor.Instructor
    timeout: int

    def __init__(
        self, model: str, *, base_url: str = "http://localhost:11434", timeout: int = 2 * 60
    ):
        """Initialise the ollama extraction model and create the instructor client."""
        self.model = model
        self.base_url = base_url
        self.timeout = timeout

        self.client = _get_client(model=model, ollama_base_url=base_url)

    def predict(self, path: Path) -> ExtractionOutput:
        """Run inference on *path* and return an ExtractionOutput with results or error info."""
        start = time.perf_counter()
        try:
            extraction, completion = _get_ollama_extraction(
                file_path=path, extraction_client=self.client, timeout=self.timeout
            )
            ok = True
            error_message = ""
        except OllamaInferenceError as ex:
            extraction, completion = None, None
            ok = False
            error_message = str(ex)

        elapsed = time.perf_counter() - start
        return ExtractionOutput(
            extraction=extraction,
            completion=completion,
            elapsed=elapsed,
            ok=ok,
            error_message=error_message,
        )


class LLamaSwapExtractionModel:
    """Extraction model that delegates inference to a llama-swap server."""

    model: str
    base_url: str
    client: instructor.Instructor
    timeout: int

    def __init__(self, model: str, *, base_url: str = LLAMA_SWAP_URL, timeout: int = 2 * 60):
        """Initialise the llama-swap extraction model and create the instructor client."""
        self.model = model
        self.base_url = base_url
        self.timeout = timeout

        self.client = instructor.from_openai(
            OpenAI(base_url=f"{base_url}/v1", api_key="not-needed"),
        )

    def predict(self, path: Path) -> ExtractionOutput:
        """Run inference on *path* and return an ExtractionOutput with results or error info."""
        start = time.perf_counter()
        try:
            extraction, completion = _get_llama_swap_extraction(
                file_path=path,
                extraction_client=self.client,
                model=self.model,
                timeout=self.timeout,
            )
            ok = True
            error_message = ""
        except InferenceError as ex:
            extraction, completion = None, None
            ok = False
            error_message = str(ex)

        elapsed = time.perf_counter() - start
        return ExtractionOutput(
            extraction=extraction,
            completion=completion,
            elapsed=elapsed,
            ok=ok,
            error_message=error_message,
        )


class ResultDetails(TypedDict):
    """Typed dictionary capturing per-sample extraction result details."""

    run: int
    idx: int
    path: str
    image_height: int
    image_width: int
    elapsed: float
    ok: bool
    error_message: str
    y_pred: float | None
    y_true: float
    completion_tokens: int | None
    prompt_tokens: int | None
    total_tokens: int | None
    reasoning_tokens: int | None

In [ ]:
def do_extractions(provider: ModelProvider, model: str, n_runs: int) -> pl.DataFrame:
    """Run *n_runs* extraction passes over the dataset and return a tidy results DataFrame."""
    match provider:
        case ModelProvider.ollama:
            estimator = OllamaExtractionModel(model=model)
        case ModelProvider.llama_swap:
            estimator = LLamaSwapExtractionModel(model=model)

    _results = []

    for _run in range(n_runs):
        print(f"run {_run + 1} / {n_runs}")

        for _idx, (_img, _path, _amount) in tqdm.tqdm(enumerate(ds), total=len(ds)):
            _o = estimator.predict(_path)

            if _o.ok:
                if _o.completion is None:
                    msg = "_completion is unexpectedly None"
                    raise ValueError(msg)

                elif _o.completion.usage is None:
                    msg = "_completion.usage is unexpectedly None"
                    raise ValueError(msg)

                if _o.completion.usage.completion_tokens_details is None:
                    msg = "_completion.usage.completion_tokens_details is unexpectedly None"
                    raise ValueError(msg)

            _r: ResultDetails = {
                "run": _run,
                "idx": _idx,
                "path": str(_path),
                "image_height": _img.height,
                "image_width": _img.width,
                "ok": _o.ok,
                "error_message": _o.error_message,
                "elapsed": _o.elapsed,
                "y_pred": _o.extraction.amount if _o.extraction else None,
                "y_true": _amount,
                "completion_tokens": _o.completion.usage.completion_tokens if _o.ok else None,  # ty: ignore[unresolved-attribute]
                "prompt_tokens": _o.completion.usage.prompt_tokens if _o.ok else None,  # ty: ignore[unresolved-attribute]
                "total_tokens": _o.completion.usage.total_tokens if _o.ok else None,  # ty: ignore[unresolved-attribute]
                "reasoning_tokens": _o.completion.usage.completion_tokens_details.reasoning_tokens  # ty: ignore[unresolved-attribute]
                if _o.ok
                else None,
            }
            _results.append(_r)

    return pl.from_dicts(_results)


if PERFORM_EXTRACTIONS:
    results_df = do_extractions(PROVIDER, MODEL, N_RUNS)
    display(results_df.head(2))  # ty: ignore[unresolved-reference]

In [ ]:
provider_dir = RESULTS_DIR / PROVIDER.value
provider_dir.mkdir(exist_ok=True)

benchmark_result_path = provider_dir / f"{MODEL.replace(':', '-')}.parquet"
benchmark_result_path

In [ ]:
if PERFORM_EXTRACTIONS:
    results_df.write_parquet(benchmark_result_path)
else:
    results_df = pl.read_parquet(benchmark_result_path)

## Analyse run results

In [ ]:
results_df.head(2)

In [ ]:
plot_df = results_df.with_columns(
    **{
        "image size": pl.col("image_height") * pl.col("image_width"),
        "amount delta": pl.col("y_pred") - pl.col("y_true"),
        "relative amount delta": (pl.col("y_pred") - pl.col("y_true")) / pl.col("y_true"),
    }
)
plot_df.head(2)

### Run -> latency

In [ ]:
(ggplot(plot_df, aes(x="run", y="elapsed", color="idx")) + geom_point())

### Image size -> latency

In [ ]:
(ggplot(plot_df, aes(x="image size", y="elapsed", color="run")) + geom_point())

### Image size -> total tokens

In [ ]:
(ggplot(plot_df, aes(x="image size", y="total_tokens", color="run")) + geom_point())

### Image size -> completion tokens

In [ ]:
(ggplot(plot_df, aes(x="image size", y="completion_tokens", color="run")) + geom_point())

### Image size -> prompt tokens

In [ ]:
(ggplot(plot_df, aes(x="image size", y="prompt_tokens", color="run")) + geom_point())

### Image size -> reasoning tokens

In [ ]:
(ggplot(plot_df, aes(x="image size", y="reasoning_tokens", color="run")) + geom_point())

### Image size -> `ok`

In [ ]:
plot_df["ok"].value_counts()

### `amount` inference correctness and stability

In [ ]:
(ggplot(plot_df, aes(x="amount delta")) + geom_histogram(binwidth=10))

In [ ]:
plot_df.group_by("run").agg(
    pl.col("amount delta").median().alias("median"), pl.col("amount delta").mean().alias("mean")
)

In [ ]:
case_consistency = (
    plot_df.group_by("idx").agg((pl.col("y_pred").n_unique() == 1).alias("consistent")).sort("idx")
)
case_consistency

In [ ]:
overall_consistency = case_consistency["consistent"].mean()
overall_consistency

### First run processing time

In [ ]:
plot_df.filter(pl.col("run") == 0)["elapsed"].mean()

In [ ]:
plot_df.filter(pl.col("run") == 0)["elapsed"].median()